In [1]:
import os
import dill
import shutil
import pandas as pd
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
import numpy as np
import random

import warnings

# 禁用所有警告
warnings.filterwarnings("ignore")

In [11]:
all_df = pd.read_csv('D:/2024BI_Journal/MIMIC_IV/Median_B/All_TS_non_49023.csv')
print(all_df.shape)

(49023, 170)


In [4]:
diags = pd.read_csv('D:/2024BI_Journal/MIMIC_IV/Median_B/All_diags_binary.csv')
diag_col = diags.columns
diag_col = diag_col[~diag_col.isin(['hadm_id'])].tolist()

In [6]:
#得到所有类别的列
# 初始化分类列表
chart_col = []
lab_col = []
proc_col = []
med_col = []

# 遍历输入列表，按前缀分类
for item in all_df.columns.tolist():
    if item.startswith('CHART'):
        chart_col.append(item)
    elif item.startswith('LAB'):
        lab_col.append(item)
    elif item.startswith('PROC'):
        proc_col.append(item)
    elif item.startswith('MED'):
        med_col.append(item)


diag_col = diags.columns
# diag_col = diag_col[~diag_col.isin(['HADM_ID'])].tolist()
diag_col = diag_col[~diag_col.isin(['hadm_id'])].tolist()

AET_col = ['ab_id_count', 'ab_id_filtered_count', 'ATE', 'ATE_filtered']

MDR_col = ['MDR_before']

BI_col = ['Acinetobacter spp.', 'Enterobacteriaceae', 'Enterococcus spp.',
       'Pseudomonas aeruginosa', 'Staphylococcus aureus', 'MDR_label']

all_feat_feature = chart_col + lab_col + proc_col + med_col# + diag_col + AET_col + MDR_col
len(all_feat_feature)

163

In [7]:
# 根据包含 'Cate' 或 'Str' 的特征来创建分类特征列表
df_cate = [feat for feat in all_feat_feature if 'Cate' in feat or 'Str' in feat]
print("Number of categorical features:", len(df_cate))

# 数值特征是 all_feat_feature 中不包含 'Cate' 或 'Str' 的特征
df_num = [feat for feat in all_feat_feature if feat not in df_cate]
print("Number of numerical features:", len(df_num))

Number of categorical features: 56
Number of numerical features: 107


In [12]:
alc_id = all_df.ICUSTAY_ID.unique()
len(alc_id)

49023

In [13]:
alc_id[0]

30000153

In [16]:
# 设置保存路径
save_path = 'D:/2024BI_Journal/MIMIC_IV/Median_B/Final/New_All_IV_minmax/'

# 遍历文件夹中的所有文件
for i in tqdm(alc_id):
    # 初始化一个空的 DataFrame 用于存储最终拼接的结果
    final_df = pd.DataFrame(columns=all_feat_feature)
    chart = pd.read_csv(f'D:/2024BI_Journal/MIMIC_IV/Final_Chart_2/{i}.csv')
    
    # 尝试读取其他文件，若不存在则设为 None
    try:
        lab = pd.read_csv(f'D:/2024BI_Journal/MIMIC_IV/Final_Lab_2/{i}.csv')
    except Exception as e:
        # 打印错误信息并跳过该文件
        #print(f"Error processing file {i}: {e}")
        lab = None
        
    try:
        proc = pd.read_csv(f'D:/2024BI_Journal/MIMIC_IV/Final_Proc_2/{i}.csv')
    except Exception as e:
        # 打印错误信息并跳过该文件
        #print(f"Error processing file {i}: {e}")
        proc = None
        
    try:
        med = pd.read_csv(f'D:/2024BI_Journal/MIMIC_IV/Final_Med_2/{i}.csv')
    except Exception as e:
        # 打印错误信息并跳过该文件
        #print(f"Error processing file {i}: {e}")
        med = None
    
    # 检查这些文件的行数是否一致
    data_frames = [chart, lab, proc, med]
    
    # 过滤掉 None 值的 DataFrame
    data_frames = [df for df in data_frames if df is not None]
    
    # 检查剩下的 DataFrame 的行数是否一致
    if all(df.shape[0] == chart.shape[0] for df in data_frames):
        # 按列拼接这些 DataFrame
        this_concat = pd.concat(data_frames, axis=1)
        
        # 将拼接后的结果加入到 final_df 中
        final_df = pd.concat([final_df, this_concat], axis=0, ignore_index=True)
    else:
        # 如果行数不同，跳过当前文件，并显示文件名
        print(f'Skipping {i}, row counts do not match.')
    
    # 用0填充所有缺失值
    final_df = final_df.fillna(0)
    
    # 归一化数值列
    scaler = MinMaxScaler()
    final_df[df_num] = scaler.fit_transform(final_df[df_num])

    label_encoders = {feature: LabelEncoder() for feature in df_cate}
    for feature, le in label_encoders.items():
        final_df[feature] = le.fit_transform(final_df[feature])
    
    # 保存最终的 DataFrame 为 CSV
    final_df.to_csv(f'{save_path}{i}.csv', index=False)

100%|██████████████████████████████████████████████████████████████████████████| 49023/49023 [1:44:03<00:00,  7.85it/s]
